## Черновик

In [1]:
import pandas as pd

from validator import validate_prices

In [2]:
df = pd.read_excel("data/Тест.Данные.xlsx", sheet_name="Данные")

In [3]:
# Первичная проверка цен конкурентов и подсчёт дублей
for column in ["Регулярная цена конкурента", "Акционная цена конкурента"]:
    raw = df[column]
    price = pd.to_numeric(raw, errors="coerce")

    print(column)
    print("  Пустых:", raw.isna().sum())
    print("  Нечисловых:", (raw.notna() & price.isna()).sum())
    print("  Неположительных:", price.le(0).sum())

key = ["СКЮ", "Город", "Конкурент"]

print("\nВсего строк:", len(df))
print("Точных дублей:", df.duplicated().sum())
print("Повторов по ключу сверх первой строки:", df.duplicated(subset=key).sum())

Регулярная цена конкурента
  Пустых: 0
  Нечисловых: 0
  Неположительных: 0
Акционная цена конкурента
  Пустых: 0
  Нечисловых: 0
  Неположительных: 0

Всего строк: 202815
Точных дублей: 31212
Повторов по ключу сверх первой строки: 33981


In [4]:
# Повторные наблюдения и изменение регулярной цены конкурента
key = ["СКЮ", "Город", "Конкурент"]
time_col = "Обновление"
price_col = "Регулярная цена конкурента"

history = df.drop_duplicates()
same_time = history.duplicated(key + [time_col], keep=False)

print("Строк с одинаковым ключом и временем:", same_time.sum())
print("Строк без времени обновления:", history[time_col].isna().sum())

history = history.loc[~same_time].dropna(subset=[time_col])
history = history.sort_values(key + [time_col])

history["Предыдущая цена"] = history.groupby(key)[price_col].shift()
history["Изменение, %"] = (
    history[price_col] / history["Предыдущая цена"] - 1
) * 100

print("Сравнений с предыдущим замером:", history["Изменение, %"].count())

columns = key + [time_col, "Предыдущая цена", price_col, "Изменение, %"]
largest_changes = history["Изменение, %"].abs().nlargest(10)
display(history.loc[largest_changes.index, columns])

Строк с одинаковым ключом и временем: 0
Строк без времени обновления: 0
Сравнений с предыдущим замером: 2769


,СКЮ,Город,Конкурент,Обновление,Предыдущая цена,Регулярная цена конкурента,"Изменение, %"
156442,(439441) Кофе растворимый EGOISTE Noir сублими...,Санкт-Петербург,Магнит,46289.246701,499.99,999.99,100.002000
155340,(864502) Сливки ВКУСНОТЕЕВО питьевые ультрапас...,Санкт-Петербург,Магнит,46289.326632,119.99,219.99,83.340278
64375,(604892) Масло оливковое COSTA D'ORO рафинир. ...,Краснодар,Магнит,46288.622708,549.00,949.99,73.040073
198931,(185746) Кофе растворимый MONARCH Ориджинал Ин...,Челябинск,Магнит,46289.358229,779.99,399.99,-48.718573
202784,(580223) Масло сливочное БЕЛЕБЕЕВСКИЙ Традицио...,Челябинск,Магнит,46289.361424,179.00,259.99,45.245810
157749,(167069) Приправа KOTANYI Черный перец горошек...,Санкт-Петербург,Магнит,46289.339132,399.99,219.99,-45.001125
155285,(078235) Ветчина ИМПЕРИЯ ВКУСА с индейкой (Рос...,Санкт-Петербург,Магнит,46289.318148,279.99,159.99,-42.858674
156771,(714088) Кофе молотый COFFEE TURCA Turkish Cof...,Санкт-Петербург,Магнит,46289.246725,219.99,309.99,40.910950
157414,(566494) Сахар РУССКИЙ САХАР белый кусковой (Р...,Санкт-Петербург,Магнит,46289.292164,89.00,125.00,40.449438
157608,(154751) Макароны MAKFA Макфики в/с гр.А (Росс...,Санкт-Петербург,Магнит,46289.340637,49.99,69.99,40.008002


In [5]:
# Проверка валидатора на исходной выгрузке

clean, excluded, summary, by_reason = validate_prices(df)

print(summary)
display(by_reason)

{'Обработано': 202815, 'Прошло проверку': 171603, 'Не прошло проверку': 31212}


Причина
Точный дубль    31212
Name: count, dtype: int64[pyarrow]